In [1]:
# Find number of athletes meeting 84th singapore open threshold marks

#1. At least 1 athlete per event per gender
#2. Except for 100m, 400m, cap at 3 athletes per event. 100m, 400m capped at 6
#3. Where top athlete is >30 yrs old (except marathon), to include next athlete as well (below 30)
#4. Where althlete qualified in 2 events, to choose the better performing one
#5. For athletes looking to do full time, to write in to SAA for special consideration
#6. Exclude SPEX carded athletes
#7. Except for marathon, age threshold cut off of 40 yrs old for top athlete
#8. No double tapping of prog - potential names in red


%load_ext autoreload
%autoreload 2

In [2]:
# Import usual modules
import pandas as pd
import csv
import math
import os
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import openpyxl
import datetime
from scipy.stats import lognorm
import re
import string
from bs4 import BeautifulSoup
import requests
import unicodedata # for removing accented characters
import datetime
import icecream as ic
import dateutil.parser as parser 
import streamlit as st
import plotly.graph_objects as go
import plotly.offline as pyo
import plotly.express as px
from plotly.subplots import make_subplots


from google.cloud import storage



In [3]:
# PRODUCTION ENVIRONMENT
# Extract timed event records

import pandas_gbq
from google.oauth2 import service_account

credentials = service_account.Credentials.from_service_account_file(
    '/Users/veesheenyuen/Desktop/DataScience/Keys/saa-analytics-7c8937b70609.json',
    
    
)

sql1="""
SELECT NAME, RESULT, TEAM, AGE, RANK AS COMPETITION_RANK, DIVISION, EVENT, DISTANCE, EVENT_CLASS, UNIQUE_ID, DOB, NATIONALITY, WIND, CATEGORY_EVENT, GENDER, COMPETITION, YEAR, REGION
FROM `saa-analytics.results.athlete_results_prod` 
WHERE RESULT!='NM' AND RESULT!='-' AND RESULT!='DNS' AND RESULT!='DNF' AND RESULT!='DNQ' AND RESULT!='DQ' AND RESULT IS NOT NULL

"""


competitors = pandas_gbq.read_gbq(sql1, project_id="saa-analytics", credentials=credentials)


Downloading: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████|


In [4]:
competitors

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,DOB,NATIONALITY,WIND,CATEGORY_EVENT,GENDER,COMPETITION,YEAR,REGION
0,Shanti Veronica Pereira,11.71,nan,nan,7.0,nan,100m,nan,nan,nan,20-Sep-96,SGP,-1.8,Sprint,Female,AtleticaGenève,2025,International
1,Shanti Veronica Pereira,23.16w,nan,nan,7.0,nan,200m,nan,nan,nan,20-Sep-96,SGP,2.4,Sprint,Female,AtleticaGenève,2025,International
2,Jun Yu Low,4.77,nan,nan,5.0,nan,Pole Vault,nan,nan,nan,21-Apr-01,SGP,nan,Jump,Male,Jan Dietvorst Memorial,2025,International
3,Tung Hon Andrew Pak,2.05,nan,nan,nan,nan,High Jump,nan,nan,nan,10-Apr-02,SGP,nan,Jump,Male,National Championships,2025,International
4,Tam Jong Hng,1.95,nan,nan,nan,nan,High Jump,nan,nan,nan,nan,SGP,nan,Jump,Male,National Championships,2025,International
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31169,Charlene Lee,8.90,NYGH,None,15,C,Triple Jump,None,None,None,None,None,None,Jump,Female,National School Games,2024,Local
31170,"Ng Jing Wen, Ariel",10.33,RGS,None,2,C,Triple Jump,None,None,None,None,None,None,Jump,Female,National School Games,2024,Local
31171,LEE GABRIEL JIN YI,14.5,SINGAPORE,21,3,None,Triple jump,None,None,None,2003,SGP,NWI,Jump,Male,Victor Saneev Memorial (Georgia),2024,International
31172,"SYED AHMED RIADH, SHARIFAH FALISHA",11.13,SINGAPORE SPORTS SCHOOL,18,None,None,Triple jump,None,None,None,2006,SGP,NWI,Jump,Female,FTKLAA State Meet,2024,International


In [5]:
# Remove special characters before mapping

athletes = competitors

for col in athletes.columns:
    athletes[col] = athletes[col].astype(str)
    athletes[col] = athletes[col].str.replace('\xa0', ' ', regex=True)
    athletes[col] = athletes[col].str.replace('[\x00-\x1f\x7f-\x9f]', '', regex=True)
    athletes[col] = athletes[col].str.replace('\r', ' ', regex=True)
    athletes[col] = athletes[col].str.replace('\n', ' ', regex=True)
    athletes[col] = athletes[col].str.strip()

# Create temporary mapped event column

athletes['MAPPED_EVENT']=''

# Correct javelin category

mask = athletes['EVENT'].str.contains(r'Javelin', na=True)
athletes.loc[mask, 'CATEGORY_EVENT'] = 'Throw'

# Throws



#mask = (athletes['EVENT'].str.contains(r'Javelin Throw|Javelin throw|Javelin', na=False) & athletes['EVENT_CLASS'].str.contains(r'600g', na=False) & athletes['GENDER'].str.contains(r'Female', na=False))
#athletes.loc[mask, 'MAPPED_EVENT'] = 'Javelin Throw'
#mask = (athletes['EVENT'].str.contains(r'Javelin Throw|Javelin throw|Javelin', na=False) & athletes['EVENT_CLASS'].str.contains(r'800g', na=False) & athletes['GENDER'].str.contains(r'Male', na=False))
#athletes.loc[mask, 'MAPPED_EVENT'] = 'Javelin Throw'
#mask = (athletes['EVENT'].str.contains(r'Javelin Throw|Javelin throw', na=False) & athletes['DIVISION'].str.contains(r'OPEN|Open', na=False))
#athletes.loc[mask, 'MAPPED_EVENT'] = 'Javelin Throw'
mask = (athletes['EVENT'].str.contains(r'Javelin|Javelin Throw', na=False))
athletes.loc[mask, 'MAPPED_EVENT'] = 'Javelin Throw'



#mask = (athletes['EVENT'].str.contains(r'Shot Put|Shot put', na=False, regex=True) & (athletes['GENDER']=='Female') & (athletes['EVENT_CLASS']=='4kg'))# there are some additional characters after Put
#athletes.loc[mask, 'MAPPED_EVENT'] = 'Shot Put'
#mask = (athletes['EVENT'].str.contains(r'Shot Put|Shot put', na=False) & (athletes['GENDER']=='Male') & (athletes['EVENT_CLASS'].str.contains(r'7.26', na=False)))# there are some additional characters after Put
#athletes.loc[mask, 'MAPPED_EVENT'] = 'Shot Put'
#mask = (athletes['EVENT'].str.contains(r'Shot Put|Shot put', na=False) & (athletes['GENDER']=='Female') & (athletes['EVENT_CLASS'].str.contains(r'4', na=False)))# there are some additional characters after Put
#athletes.loc[mask, 'MAPPED_EVENT'] = 'Shot Put'

#mask = (athletes['EVENT'].str.contains(r'Shot Put|Shot put', na=False) & (athletes['DIVISION'].str.contains(r'OPEN|Open', na=False)))# there are some additional characters after Put
#athletes.loc[mask, 'MAPPED_EVENT'] = 'Shot Put'

mask = athletes['EVENT'].str.contains(r'Shot Put|Shot put', na=False)# there are some additional characters after Put
athletes.loc[mask, 'MAPPED_EVENT'] = 'Shot Put'



#mask = (athletes['EVENT'].str.contains(r'Hammer Throw|Hammer throw', na=False) & athletes['EVENT_CLASS'].str.contains(r'7.26kg', na=False))
#athletes.loc[mask, 'MAPPED_EVENT'] = 'Hammer Throw'
#mask = (athletes['EVENT'].str.contains(r'Hammer Throw|Hammer throw', na=False) & athletes['EVENT_CLASS'].str.contains(r'4.00kg', na=False))
#athletes.loc[mask, 'MAPPED_EVENT'] = 'Hammer Throw'
mask = (athletes['EVENT'].str.contains(r'Hammer Throw|Hammer throw', na=False) & (athletes['DIVISION'].str.contains(r'OPEN|Open', na=False)))# there are some additional characters after Put
athletes.loc[mask, 'MAPPED_EVENT'] = 'Hammer Throw'
ask = athletes['EVENT'].str.contains(r'Hammer Throw|Hammer throw', na=False) # there are some additional characters after Put
athletes.loc[mask, 'MAPPED_EVENT'] = 'Hammer Throw'



#mask = ((athletes['EVENT'].str.contains(r'Discus\sThrow\s\(1kg\)', na=False)) & (athletes['GENDER']=='Female'))
#athletes.loc[mask, 'MAPPED_EVENT'] = 'Discus Throw'

#mask = ((athletes['EVENT'].str.contains(r'Discus\s\(1\.00kg\)', na=False))  & (athletes['GENDER']=='Female'))
#athletes.loc[mask, 'MAPPED_EVENT'] = 'Discus Throw'


mask = athletes['EVENT'].str.contains(r'Discus\sThrow', na=False)
athletes.loc[mask, 'MAPPED_EVENT'] = 'Discus Throw'
#mask = ((athletes['EVENT'].str.contains(r'Discus\sThrow\s\(1kg\)', na=False)) & (athletes['GENDER']=='Female'))
#athletes.loc[mask, 'MAPPED_EVENT'] = 'Discus Throw'
mask = (athletes['EVENT'].str.contains(r'Discus Throw|Discus|Discus throw', na=False) & athletes['EVENT_CLASS'].str.contains(r'2kg|2.00kg', na=False) & athletes['GENDER'].str.contains(r'Male', na=False))
athletes.loc[mask, 'MAPPED_EVENT'] = 'Discus Throw'
mask = (athletes['EVENT'].str.contains(r'Discus Throw|Discus|Discus throw', na=False) & athletes['EVENT_CLASS'].str.contains(r'1kg|1.00kg', na=False) & athletes['GENDER'].str.contains(r'Female', na=False))
athletes.loc[mask, 'MAPPED_EVENT'] = 'Discus Throw'

#mask = (athletes['EVENT'].str.contains(r'Discus Throw|Discus throw', na=False) & athletes['REGION'].str.contains(r'International', na=False))
#athletes.loc[mask, 'MAPPED_EVENT'] = 'Discus Throw'
mask = (athletes['EVENT'].str.contains(r'Discus Throw|Discus throw', na=False) & athletes['DIVISION'].str.contains(r'OPEN|Open', na=False))
athletes.loc[mask, 'MAPPED_EVENT'] = 'Discus Throw'
mask = (athletes['EVENT'].str.contains(r'Discus Throw|Discus throw', na=False) & athletes['DIVISION'].str.contains(r'None', na=False) & athletes['EVENT_CLASS'].str.contains(r'None', na=False))
athletes.loc[mask, 'MAPPED_EVENT'] = 'Discus Throw'

mask = (athletes['EVENT'].str.contains(r'Discus', na=False))
athletes.loc[mask, 'MAPPED_EVENT'] = 'Discus Throw'


mask = (athletes['DIVISION'].str.contains(r'A', na=False))
athletes.loc[mask, 'AGE'] = '18.5'

mask = (athletes['DIVISION'].str.contains(r'B', na=False))
athletes.loc[mask, 'AGE'] = '16'

mask = (athletes['DIVISION'].str.contains(r'C', na=False))
athletes.loc[mask, 'AGE'] = '13.5'






In [6]:
athletes = athletes[athletes['CATEGORY_EVENT']=='Throw']

In [7]:
# Exclude foreigners from MALAYSIA, THAILAND etc.

df_selected = athletes[(athletes['TEAM']!='Malaysia')&(athletes['TEAM']!='THAILAND')&(athletes['TEAM']!='China') 
                       &(athletes['TEAM']!='South Korea')&(athletes['TEAM']!='Laos') 
                       &(athletes['TEAM']!='Philippines')&(athletes['TEAM']!='Piboonbumpen Thailand') 
                       &(athletes['TEAM']!='Chinese Taipei')&(athletes['TEAM']!='Gurkha Contingent') 
                       &(athletes['TEAM']!='Australia')&(athletes['TEAM']!='Piboonbumpen Thailand') 
                       &(athletes['TEAM']!='Hong Kong')&(athletes['TEAM']!='PERAK')&(athletes['TEAM']!='Sri Lanka') 
                       &(athletes['TEAM']!='Indonesia')&(athletes['TEAM']!='THAILAND')&(athletes['TEAM']!='MALAYSIA') 
                       &(athletes['TEAM']!='PHILIPPINES') & (athletes['TEAM']!='SOUTH KOREA')&(athletes['TEAM']!='Waseda') 
                       &(athletes['TEAM']!='LAOS')&(athletes['TEAM']!='CHINESE TAIPEI')
                       &(athletes['TEAM']!='INDIA')&(athletes['TEAM']!='Hong Kong, China')&(athletes['TEAM']!='AIC JAPAN')] 


In [8]:
df_selected = df_selected[df_selected['DIVISION']!='Masters']

In [9]:
df_selected.reset_index(inplace=True, drop=True)

In [10]:
df_selected

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,DOB,NATIONALITY,WIND,CATEGORY_EVENT,GENDER,COMPETITION,YEAR,REGION,MAPPED_EVENT
0,"Chong Bin Mohd Isham, Akid",32.40m,FAC,34,2,Open,Hammer Throw,0,(7.26kg),A512E90,24/11/90,nan,,Throw,Male,54th SA Inter Club Championships 2024,2024,Local,Hammer Throw
1,"Ng, Joash",1.89m,Team M&J,5,10,U7,Shot Put,0,1kg Medicine,nan,11/1/19,nan,,Throw,Male,11th Club ZOOM Kindred Spirit 2024 & PEERS,2024,Local,Shot Put
2,"Franz, Alexander",3.43m,Club ZOOM,5,3,U7,Shot Put,0,1kg Medicine,nan,26/4/19,nan,,Throw,Male,11th Club ZOOM Kindred Spirit 2024 & PEERS,2024,Local,Shot Put
3,"Lee, Tzu Yun",53.18m,Club Zoom,30,1,Open,Discus Throw,0,,nan,24/11/94,nan,,Throw,Male,84th Singapore Open Track & Field,2024,Local,Discus Throw
4,"YEE WAI TENG, MELISSA",10.60m,SINGAPORE,26,2,Open,Shot Put,0,,nan,8/8/98,nan,,Throw,Female,84th Singapore Open Track & Field,2024,Local,Shot Put
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1703,"Sheik Alau'ddin, Sheik Fayz",9.40m,Republic Polytechnic,0,3,Open,Shot Put,0,None,nan,nan,nan,None,Throw,Male,IVP Track & Field Championships 2025,2025,Local,Shot Put
1704,"Tay, Arvin",10.07m,Singapore Management Universit,0,2,Open,Shot Put,0,None,nan,nan,nan,None,Throw,Male,IVP Track & Field Championships 2025,2025,Local,Shot Put
1705,"Josiah, Chan",7.03m,Singapore Polytechnic,0,7,Open,Shot Put,0,None,nan,nan,nan,None,Throw,Male,IVP Track & Field Championships 2025,2025,Local,Shot Put
1706,"Subba, Sugam",7.17m,Singapore Polytechnic,0,6,Open,Shot Put,0,None,nan,nan,nan,None,Throw,Male,IVP Track & Field Championships 2025,2025,Local,Shot Put


In [11]:
os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/Throws Program/')


df_selected.to_csv('athletes_throws_mapped.csv', sep=',', encoding='utf-8-sig', index=False)


In [12]:
# Converts any time format into seconds

def convert_time(i, string, metric):

    global output
    
    l=['discus', 'throw', 'jump', 'vault', 'shot']
        
    string=string.lower()
    
   # print('metric', metric)
    
    try:
        
        if 'w' in metric:  # skip marks with illegal wind speeds
            
            print('W', metric)
            
            output=''
            
        else:
            
    
            if any(s in string for s in l)==True:
            
                if 'm' in metric:
            
                    metric=metric.replace('m', '')
                    output=float(str(metric))
            
                elif 'GR' in metric:
            
                    metric=metric.replace('GR', '')
                    output=float(str(metric))
                
          #  elif 'w' in metric:
                
               
           #     metric1=metric.replace('w', '')
                
           #     output=float(str(metric1))
                
                else:
    
                    output=float(str(metric))
        
        #elif 'w' in metric:
            
                
         #       metric1=metric.replace('w', '')
                             
         #       output=float(str(metric1))
                
                
        
            else:
        
                searchstring = ":"
                searchstring2 = "."
                substring=str(metric)
                count = substring.count(searchstring)
                count2 = substring.count(searchstring2)
            
                if count==0:
                
                    output=float(substring)
            
            
      #      elif 'marathon' in string and count==2:
                
      #          print('HERE', metric)
                
      #          metric = metric.replace(":", ".", 2)
                
      #          print('METRIC', metric)
                
       #         h,m,s = metric.split(':')            

       #         output = float(datetime.timedelta(hours=int(h),minutes=int(m),seconds=float(s)).total_seconds())

               
             
                elif (type(metric)==datetime.time or type(metric)==datetime.datetime):
                
                                                
                    time=str(metric)
                    h, m ,s = time.split(':')
                    output = float(datetime.timedelta(hours=int(h),minutes=int(m),seconds=float(s)).total_seconds())
            
                                
                elif (count==1 and count2==1):
            
                    m,s = metric.split(':')
                    output = float(datetime.timedelta(minutes=int(m),seconds=float(s)).total_seconds())
                     
                elif (count==1 and count2==2):
                
            
                    metric = metric.replace(".", ":", 1)
            
                    h,m,s = metric.split(':')            
                    output = float(datetime.timedelta(hours=int(h),minutes=int(m),seconds=float(s)).total_seconds())
                
        
                elif (count==2 and count2==0):
                
            
                    h,m,s = metric.split(':')
                    output = float(datetime.timedelta(hours=int(h),minutes=int(m),seconds=float(s)).total_seconds())
  
            

    except:
        
        pass
                
    return output

In [13]:
for i in range(len(df_selected)):
        
    rowIndex = df_selected.index[i]

    input_string=df_selected.iloc[rowIndex,18]
    
    metric=df_selected.iloc[rowIndex,1]
    
    if metric==None:
        continue
        
    out = convert_time(i, input_string, metric)
    
    print(rowIndex, input_string, out)
     
    df_selected.loc[rowIndex, 'Metric'] = out

0 Hammer Throw 32.4
1 Shot Put 1.89
2 Shot Put 3.43
3 Discus Throw 53.18
4 Shot Put 10.6
5 Discus Throw 35.41
6 Shot Put 11.44
7 Shot Put 11.03
8 Discus Throw 37.84
9 Javelin Throw 43.78
10 Shot Put 11.63
11 Javelin Throw 32.4
12 Discus Throw 25.5
13 Shot Put 8.39
14 Shot Put 7.4
15 Javelin Throw 27.17
16 Discus Throw 15.6
17 Shot Put 6.15
18 Shot Put 10.86
19 Shot Put 10.58
20 Shot Put 7.91
21 Javelin Throw 52.9
22 Discus Throw 35.22
23 Shot Put 9.82
24 Discus Throw 32.28
25 Shot Put 11.28
26 Javelin Throw 56.64
27 Javelin Throw 54.1
28 Javelin Throw 29.75
29 Shot Put 10.64
30 Hammer Throw 32.19
31 Javelin Throw 53.81
32 Discus Throw 31.39
33 Javelin Throw 61.54
34 Discus Throw 37.13
35 Shot Put 37.13
36 Javelin Throw 30.06
37 Javelin Throw 22.89
38 Shot Put 11.58
39 Discus Throw 30.48
40 Javelin Throw 26.18
41 Javelin Throw 24.25
42 Hammer Throw 21.33
43 Discus Throw 25.51
44 Javelin Throw 25.59
45 Javelin Throw 37.96
46 Javelin Throw 33.67
47 Hammer Throw 12.81
48 Discus Throw 13.64

In [14]:
df_selected

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,DOB,NATIONALITY,WIND,CATEGORY_EVENT,GENDER,COMPETITION,YEAR,REGION,MAPPED_EVENT,Metric
0,"Chong Bin Mohd Isham, Akid",32.40m,FAC,34,2,Open,Hammer Throw,0,(7.26kg),A512E90,24/11/90,nan,,Throw,Male,54th SA Inter Club Championships 2024,2024,Local,Hammer Throw,32.40
1,"Ng, Joash",1.89m,Team M&J,5,10,U7,Shot Put,0,1kg Medicine,nan,11/1/19,nan,,Throw,Male,11th Club ZOOM Kindred Spirit 2024 & PEERS,2024,Local,Shot Put,1.89
2,"Franz, Alexander",3.43m,Club ZOOM,5,3,U7,Shot Put,0,1kg Medicine,nan,26/4/19,nan,,Throw,Male,11th Club ZOOM Kindred Spirit 2024 & PEERS,2024,Local,Shot Put,3.43
3,"Lee, Tzu Yun",53.18m,Club Zoom,30,1,Open,Discus Throw,0,,nan,24/11/94,nan,,Throw,Male,84th Singapore Open Track & Field,2024,Local,Discus Throw,53.18
4,"YEE WAI TENG, MELISSA",10.60m,SINGAPORE,26,2,Open,Shot Put,0,,nan,8/8/98,nan,,Throw,Female,84th Singapore Open Track & Field,2024,Local,Shot Put,10.60
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1703,"Sheik Alau'ddin, Sheik Fayz",9.40m,Republic Polytechnic,0,3,Open,Shot Put,0,None,nan,nan,nan,None,Throw,Male,IVP Track & Field Championships 2025,2025,Local,Shot Put,9.40
1704,"Tay, Arvin",10.07m,Singapore Management Universit,0,2,Open,Shot Put,0,None,nan,nan,nan,None,Throw,Male,IVP Track & Field Championships 2025,2025,Local,Shot Put,10.07
1705,"Josiah, Chan",7.03m,Singapore Polytechnic,0,7,Open,Shot Put,0,None,nan,nan,nan,None,Throw,Male,IVP Track & Field Championships 2025,2025,Local,Shot Put,7.03
1706,"Subba, Sugam",7.17m,Singapore Polytechnic,0,6,Open,Shot Put,0,None,nan,nan,nan,None,Throw,Male,IVP Track & Field Championships 2025,2025,Local,Shot Put,7.17


In [15]:
def length(string):

    B = ''
    year = ''

    try:

        length = len(string)

        if length == 2:

            string = '19' + string

        elif length == 1:

            string = ''

        else:

            pass

        if string is not None or len(string) != 1:

            B = parser.parse(string, dayfirst=True)
                        
    except:

        pass

    return B


df_selected['DOB_new'] = df_selected['DOB'].apply(length)



#B = parser.parse("10-09-2021", dayfirst = True)

In [16]:
type(df_selected['DOB_new'])

pandas.core.series.Series

In [17]:
df_selected['DOB_new'] = pd.to_datetime(df_selected['DOB_new'], errors='coerce')


df_selected['year_extract']=df_selected['DOB_new'].dt.strftime('%Y')
#df_selected['year_extract'] = df_selected['year_extract'].replace(regex=r'20', value='19')

#df_selected['age_extract'].apply(np.sign).replace({-1: 'down', 1: 'up', 0: 'zero'})


df_selected['year_extract'] = pd.to_numeric(df_selected['year_extract'])

In [18]:
df_selected['age_extract'] = 2024 - df_selected['year_extract']



In [19]:
def age(number):  # correct negative age numbers
    
    if number<0:
        
        number+=100
        
    return number


df_selected['age_extract'] = df_selected['age_extract'].apply(age)


In [20]:
df_selected.tail(50)

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,CATEGORY_EVENT,GENDER,COMPETITION,YEAR,REGION,MAPPED_EVENT,Metric,DOB_new,year_extract,age_extract
1658,"Low, Jun Jie Jonathan",44.41m,National University Singapore,0,1,Open,Discus Throw,0,None,nan,...,Throw,Male,IVP Track & Field Championships 2025,2025,Local,Discus Throw,44.41,NaT,NaN,NaN
1659,"Qin, Yi",25.18m,National University Singapore,0,8,Open,Discus Throw,0,None,nan,...,Throw,Male,IVP Track & Field Championships 2025,2025,Local,Discus Throw,25.18,NaT,NaN,NaN
1660,"Chandramohan Meena, Girish",31.03m,Singapore University of Social,0,4,Open,Discus Throw,0,None,nan,...,Throw,Male,IVP Track & Field Championships 2025,2025,Local,Discus Throw,31.03,NaT,NaN,NaN
1661,"Sheik Alau'ddin, Sheik Fayz",40.70m,Republic Polytechnic,0,2,Open,Discus Throw,0,None,nan,...,Throw,Male,IVP Track & Field Championships 2025,2025,Local,Discus Throw,40.70,NaT,NaN,NaN
1662,"Mook, He Jun",36.50m,Singapore Management Universit,0,3,Open,Discus Throw,0,None,nan,...,Throw,Male,IVP Track & Field Championships 2025,2025,Local,Discus Throw,36.50,NaT,NaN,NaN
1663,"Josiah, Chan",26.70m,Singapore Polytechnic,0,6,Open,Discus Throw,0,None,nan,...,Throw,Male,IVP Track & Field Championships 2025,2025,Local,Discus Throw,26.70,NaT,NaN,NaN
1664,"Ng, Bryan",23.26m,Singapore Polytechnic,0,10,Open,Discus Throw,0,None,nan,...,Throw,Male,IVP Track & Field Championships 2025,2025,Local,Discus Throw,23.26,NaT,NaN,NaN
1665,"Chan, Cara",7.61m,Ngee Ann Polytechnic,0,3,Open,Shot Put,0,None,nan,...,Throw,Female,IVP Track & Field Championships 2025,2025,Local,Shot Put,7.61,NaT,NaN,NaN
1666,"Phua, Kia Ern, Jaydene",8.07m,National University Singapore,0,2,Open,Shot Put,0,None,nan,...,Throw,Female,IVP Track & Field Championships 2025,2025,Local,Shot Put,8.07,NaT,NaN,NaN
1667,"Wan, Kaixin Clarice",8.63m,National University Singapore,0,1,Open,Shot Put,0,None,nan,...,Throw,Female,IVP Track & Field Championships 2025,2025,Local,Shot Put,8.63,NaT,NaN,NaN


In [21]:
# If NSG event then choose AGE otherwise choose age_extract

condition1=df_selected['COMPETITION']=='National School Games'
#condition2=((df['CATEGORY_EVENT']=='Jump')|(df['CATEGORY_EVENT']=='Throw'))
#condition3=df['SEED_CONV']<df['RESULT_CONV']
#condition4=~((df['CATEGORY_EVENT']=='Jump')|(df['CATEGORY_EVENT']=='Throw'))


df_selected['age_extract']=df_selected['AGE'].where((condition1), df_selected['age_extract'].values)


In [22]:
os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/Throws Program/')


df_selected.to_csv('athletes_throws_mapped.csv', sep=',', encoding='utf-8-sig', index=False)


In [23]:
df_male = df_selected[((df_selected['CATEGORY_EVENT']=='Throw') & (df_selected['GENDER']=='Male'))]

In [26]:
df_male = df_male.rename(columns={'Metric': 'Distance(m)', 'age_extract': 'Age(yrs)'})


In [27]:
fig = px.scatter(df_male, x="Age(yrs)", y="Distance(m)", color='MAPPED_EVENT', symbol ='MAPPED_EVENT', color_discrete_sequence=['red', 'blue', 'yellow', 'green'])
fig.add_hline(y=66.2, line_width=3, line_dash="dash", line_color="green", annotation_text="Javelin Throw Male")
fig.add_hline(y=50.02, line_width=3, line_dash="dash", line_color="yellow", annotation_text="Discus Throw Male")
fig.add_hline(y=17.3, line_width=3, line_dash="dash", line_color="blue", annotation_text="Shot Put Male")
fig.add_hline(y=59.76, line_width=3, line_dash="dash", line_color="red", annotation_text="Hammer Throw Male")


fig.show()

In [2690]:
df_female = df_selected[((df_selected['CATEGORY_EVENT']=='Throw') & (df_selected['GENDER']=='Female'))]

fig = px.scatter(df_female, x="age_extract", y="Metric", color='MAPPED_EVENT', symbol ='MAPPED_EVENT', color_discrete_sequence=['red', 'blue', 'green'])
fig.add_hline(y=40.59, line_width=3, line_dash="dash", line_color="pink", annotation_text="Javelin Female")

fig.show()

In [2691]:
import pandas_gbq
from google.oauth2 import service_account


credentials = service_account.Credentials.from_service_account_file(
    '/Users/veesheenyuen/Desktop/DataScience/Keys/saa-analytics-7c8937b70609.json',
)

sql="""
SELECT YEAR, EVENT, SUB_EVENT, GENDER, NAME, RESULT, RANK, CATEGORY_EVENT, COMPETITION, STAGE, HEAT
FROM `saa-analytics.benchmarks.saa_benchmarks_prod`
WHERE YEAR='2023' AND COMPETITION='Southeast Asian Games' AND (RANK='3' OR RANK='3.0')
"""

SEAG = pandas_gbq.read_gbq(sql, project_id="saa-analytics", credentials=credentials)



Downloading: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████|


In [2692]:
SEAG

,YEAR,EVENT,SUB_EVENT,GENDER,NAME,RESULT,RANK,CATEGORY_EVENT,COMPETITION,STAGE,HEAT
0,2023,4 x 100m,None,Male,"{0: '\xa0', 1: ' Jonathan Nyepa, Khairul Hafiz...",39.36,3,Relay,Southeast Asian Games,None,None
1,2023,4 x 400m,None,Male,"{0: '\xa0', 1: ' Muhammad Firdaus Bin Mohamad ...",03:08.8,3,Relay,Southeast Asian Games,None,None
2,2023,4 x 100m,None,Female,"{0: '\xa0', 1: ' Azreen Nabila Alias, Nur Afri...",44.58,3,Relay,Southeast Asian Games,None,None
3,2023,4 x 400m,None,Female,"{0: '\xa0', 1: ' Sukanya Janchaona, Benny Nont...",03:39.3,3,Relay,Southeast Asian Games,None,None
4,2023,400m,None,Male,Frederick Ramirez,46.63,3,Sprint,Southeast Asian Games,None,None
...,...,...,...,...,...,...,...,...,...,...,...
75,2023,100m Hurdles,None,Female,Natchaya Chowpakpang,14.23,3,Hurdles,Southeast Asian Games,Heats,Heat 2
76,2023,Marathon,None,Male,Nguyen Thanh Hoang,2:35:49,3.0,Marathon,Southeast Asian Games,None,None
77,2023,Marathon,None,Female,Christine Organiza Hallasgo,2:50:27,3.0,Marathon,Southeast Asian Games,None,None
78,2023,20km Race Walk,None,Female,Kotchaphon Tangsrivong,1:57:11,3.0,20km Race Walk,Southeast Asian Games,None,None


In [2693]:
benchmarks = SEAG[SEAG['CATEGORY_EVENT']=='Throw']

In [2694]:
benchmarks

,YEAR,EVENT,SUB_EVENT,GENDER,NAME,RESULT,RANK,CATEGORY_EVENT,COMPETITION,STAGE,HEAT
10,2023,Shot Put,None,Male,Muhammad Ziyad Zolkefli,17.3,3,Throw,Southeast Asian Games,None,None
20,2023,Discus Throw,None,Male,Bandit Singhatongkul,50.02,3,Throw,Southeast Asian Games,None,None
21,2023,Javelin Throw,None,Male,Agustinus Abadi Ndiken,66.2,3,Throw,Southeast Asian Games,None,None
27,2023,Discus Throw,None,Female,Le Thi Cam Dung,45.08,3,Throw,Southeast Asian Games,None,None
35,2023,Shot Put,Multievents,Female,Norliana Kamaruddin,10.59,3,Throw,Southeast Asian Games,None,None
36,2023,Javelin Throw,Multievents,Female,Nguyen Linh Na,40.59,3,Throw,Southeast Asian Games,None,None
43,2023,Hammer Throw,None,Male,Sadat Marzuki Ajisan,59.76,3,Throw,Southeast Asian Games,None,None
49,2023,Shot Put,None,Female,Nani Shahirah Maryata,14.44,3,Throw,Southeast Asian Games,None,None
50,2023,Hammer Throw,None,Female,Nurul Hidayah Lukman,49.61,3,Throw,Southeast Asian Games,None,None
51,2023,Javelin Throw,None,Female,Evalyn Palabrica,48.31,3,Throw,Southeast Asian Games,None,None


In [2695]:
# Create a line chart using Plotly graph objects
fig = go.Figure()

# Variable 1
fig.add_trace(go.Scatter(
    x=df_selected['age_extract'],  # x-axis
    y=df_selected['Metric'],  # y-axis
    mode='lines+markers',  # Connect data points with lines and add markers
    name='Line1',  # Name in the legend

    marker=dict(
        symbol='square',    # Change the marker symbol to a circle
        size=10,            # Set the marker size
        color='red'),       # Set the marker color

    line=dict(
        color='blue',        # Change the line color to red
        width=5)             # Set the line width
))